# 00 — Extracción y exploración inicial

Este notebook extrae los archivos comprimidos de los chats de Conti e inspecciona su formato real antes de construir los parsers.
Todos estos pasos se pueden saltar si ya se tiene extraida data_Vruto. Quedan las casillas como histórico de lo que hice en su día para comenzar a tratar los datos. 

## 0. Prerequisitos

No necesario si ya están extraidos.

Ejecutar en terminal si no está instalado:
```bash
sudo apt install p7zip-full
```

In [1]:
# Importamos "subprocess", que permite ejecutar comandos del sistema operativo
# desde Python. Es como abrir una terminal y escribir un comando, pero desde código.
# "shutil" tiene utilidades para trabajar con archivos y el sistema de archivos.
# "os" da acceso a funciones del sistema operativo (como variables de entorno, rutas, etc.)
import subprocess, shutil, os

# Comprobamos si el programa "7z" (el descompresor 7-Zip) está instalado en el sistema.
# shutil.which('7z') busca el ejecutable en las rutas del sistema, igual que el comando
# "which 7z" en la terminal de Linux. Si no lo encuentra, devuelve None.
if not shutil.which('7z'):
    # Si 7z no está instalado, lanzamos un error para detener el notebook aquí.
    # EnvironmentError es el tipo de error adecuado para indicar que falta
    # un programa del sistema.
    raise EnvironmentError('p7zip-full no está instalado. Ejecuta: sudo apt install p7zip-full')

# Si llegamos aquí, 7z está disponible. Imprimimos su ruta para confirmarlo.
print('7z disponible:', shutil.which('7z'))

OSError: p7zip-full no está instalado. Ejecuta: sudo apt install p7zip-full

## 1. Rutas

In [2]:
# Path es una clase de Python para manejar rutas de archivos y carpetas.
# Es mucho más cómoda que usar cadenas de texto: podemos unir carpetas con /,
# verificar si un archivo existe, leer su contenido, etc.
from pathlib import Path

# Definimos dónde están los datos brutos originales (el archivo ZIP con los chats de Conti).
# Adapta esta ruta a donde tengas los datos brutos de Ransomware.
# Convención del repo: pon los ZIPs/carpetas en ContiLeaks/data_bruto/
RANSOMWARE_DIR = Path('../data_Vruto/ContiLeaks/raw')

# La ruta completa al archivo ZIP que contiene los chats.
# El operador "/" de Path une directorios y archivos como si fuera una barra en Linux.
CONTI_ZIP      = RANSOMWARE_DIR / 'Conti_Chats_2022.zip'

# Los datos ya vienen extraídos en data_Vruto/ContiLeaks/raw (no hace falta extraer
# nada más), así que usamos la misma carpeta como destino de referencia.
RAW_DIR        = RANSOMWARE_DIR

# Creamos la carpeta de destino si no existe todavía.
# parents=True → crea también las carpetas intermedias que falten (como "mkdir -p")
# exist_ok=True → si la carpeta ya existe, no da error
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Imprimimos las rutas para verificar que son correctas antes de continuar.
print('ZIP origen:', CONTI_ZIP)
print('Directorio destino:', RAW_DIR.resolve())  # .resolve() muestra la ruta absoluta completa
print('ZIP existe:', CONTI_ZIP.exists())           # Verifica que el archivo ZIP realmente existe

ZIP origen: ../data_Vruto/ContiLeaks/raw/Conti_Chats_2022.zip
Directorio destino: /home/drjekyll/FearOfTheDark/bloque5_ransomware/data/raw
ZIP existe: False


## 2. Extraer Conti_Chats_2022.zip (contiene 3 archivos .7z)

In [3]:
# Extraer el ZIP exterior usando el programa 7z desde Python.
# subprocess.run() ejecuta un comando del sistema operativo y espera a que termine.
# Le pasamos los argumentos como una lista de strings, igual que si los escribieras
# en la terminal: "7z x <archivo.zip> -o<destino> -y"
#   "x"           → modo "extraer" (extract)
#   str(CONTI_ZIP) → ruta al archivo ZIP como texto
#   f'-o{RAW_DIR}' → carpeta de destino (la opción -o no lleva espacio antes de la ruta)
#   '-y'           → responde "sí" automáticamente a cualquier pregunta (sobrescribir, etc.)
# capture_output=True → captura lo que imprime el programa para poder mostrarlo nosotros
# text=True         → devuelve la salida como texto (str) en vez de bytes
result = subprocess.run(
    ['7z', 'x', str(CONTI_ZIP), f'-o{RAW_DIR}', '-y'],
    capture_output=True, text=True
)

# Imprimimos la salida del programa. Si es muy larga (más de 2000 caracteres),
# mostramos solo los últimos 2000 para no saturar la pantalla.
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)

# Si el código de retorno es diferente de 0, algo salió mal.
# En Linux/Mac, los programas devuelven 0 cuando terminan bien y otro número si hay error.
if result.returncode != 0:
    print('STDERR:', result.stderr)

FileNotFoundError: [Errno 2] No such file or directory: '7z'

In [5]:
# Ver qué archivos hay en raw/ después de extraer el ZIP.
# RAW_DIR.rglob('*') busca TODOS los archivos y carpetas dentro de raw/, recursivamente.
# sorted() los ordena alfabéticamente para que la lista sea más fácil de leer.
for p in sorted(RAW_DIR.rglob('*')):
    # p.stat().st_size devuelve el tamaño del archivo en bytes.
    # Si es una carpeta (p.is_dir()), ponemos 0 como tamaño porque las carpetas
    # no tienen tamaño propio (solo lo tienen sus contenidos).
    size = p.stat().st_size if p.is_file() else 0

    # Imprimimos una línea por cada elemento.
    # "DIR" si es carpeta, o el tamaño en KB si es un archivo.
    # p.relative_to(RAW_DIR) muestra la ruta relativa (sin el prefijo largo de RAW_DIR)
    # para que sea más fácil de leer.
    print(f'{"DIR" if p.is_dir() else f"{size/1024:.0f} KB":>10}  {p.relative_to(RAW_DIR)}')

## 3. Extraer los 3 archivos .7z internos

In [6]:
# Buscamos todos los archivos con extensión .7z dentro de RAW_DIR.
# Después de extraer el ZIP exterior, encontramos 3 archivos .7z (uno por fuente de datos).
# list() convierte el generador de rglob en una lista para poder reutilizarla varias veces.
sevenz_files = list(RAW_DIR.rglob('*.7z'))

# Mostramos cuántos archivos .7z encontramos y sus nombres, para verificar.
print(f'Archivos .7z encontrados: {len(sevenz_files)}')
for f in sevenz_files:
    print(' -', f.name)

Archivos .7z encontrados: 0


In [7]:
# Extraemos cada uno de los 3 archivos .7z en su propia subcarpeta.
# Esto organiza los datos de cada fuente en su propio directorio.
for archive in sevenz_files:
    # Creamos una carpeta de destino con el mismo nombre que el archivo .7z (sin extensión).
    # Por ejemplo: "Conti Chat Logs 2020.7z" → carpeta "Conti Chat Logs 2020/"
    dest = RAW_DIR / archive.stem
    dest.mkdir(exist_ok=True)  # exist_ok=True evita error si la carpeta ya existe

    # Ejecutamos 7z para extraer el archivo .7z en la carpeta de destino.
    # Los argumentos son los mismos que antes: x (extraer), ruta, -o (destino), -y (sí a todo).
    result = subprocess.run(
        ['7z', 'x', str(archive), f'-o{dest}', '-y'],
        capture_output=True, text=True
    )

    # Determinamos si la extracción fue exitosa revisando el código de retorno.
    # returncode == 0 significa éxito en Linux.
    status = 'OK' if result.returncode == 0 else 'ERROR'

    # Imprimimos una línea indicando qué archivo se extrajo y a qué carpeta.
    print(f'[{status}] {archive.name} → {dest.name}/')

    # Si hubo error, mostramos los primeros 300 caracteres del mensaje de error.
    if result.returncode != 0:
        print('  STDERR:', result.stderr[:300])

## 4. Inventario completo de archivos extraídos

In [8]:
# Importamos pandas para crear una tabla con el inventario de todos los archivos extraídos.
# Un DataFrame de pandas es como una hoja de Excel: tiene filas, columnas con nombre,
# y podemos filtrar, ordenar, agrupar, etc.
import pandas as pd

# Lista donde iremos acumulando la información de cada archivo.
inventory = []

# Recorremos todos los archivos en RAW_DIR y sus subcarpetas.
for p in sorted(RAW_DIR.rglob('*')):
    # Solo procesamos archivos (no carpetas) y saltamos los propios .7z y .zip
    # porque ya sabemos qué son: son los comprimidos que acabamos de extraer.
    if p.is_file() and p.suffix.lower() not in ('.7z', '.zip'):
        inventory.append({
            # p.relative_to(RAW_DIR) muestra la ruta sin el prefijo largo de RAW_DIR
            'archivo': p.relative_to(RAW_DIR),
            # La extensión del archivo (por ejemplo ".json")
            'extension': p.suffix.lower(),
            # Tamaño en megabytes, redondeado a 2 decimales.
            # p.stat().st_size devuelve el tamaño en bytes; dividimos entre 1024^2 para pasar a MB.
            'tamaño_MB': round(p.stat().st_size / 1024**2, 2)
        })

# Convertimos la lista de diccionarios en un DataFrame de pandas.
# Cada diccionario se convierte en una fila de la tabla.
df_inv = pd.DataFrame(inventory)

# display() muestra el DataFrame con formato bonito en el notebook (mejor que print()).
display(df_inv)

""


## 5. Inspeccionar las primeras líneas de cada archivo

In [9]:
# Inspeccionamos las primeras líneas de cada archivo para entender su formato.
# Esto nos ayuda a decidir cómo escribir el parser (el código que los lee).
# Antes de escribir código que procese los datos, es importante ver cómo son "a ojo".

# df_inv.iterrows() nos da cada fila del inventario como un par (índice, fila).
# Usamos _ para el índice porque no nos interesa el número de fila.
for _, row in df_inv.iterrows():
    # Construimos la ruta completa al archivo combinando el directorio base
    # con la ruta relativa que guardamos en el inventario.
    fpath = RAW_DIR / row['archivo']

    # Imprimimos una línea separadora y el nombre del archivo antes de mostrar su contenido.
    print(f"\n{'='*60}")
    print(f"ARCHIVO: {row['archivo']}  ({row['tamaño_MB']} MB)")
    print('='*60)

    try:
        # Abrimos el archivo en modo lectura con codificación UTF-8.
        # errors='replace' hace que los caracteres raros (cirílico, etc.) no causen error
        # sino que se reemplacen por un símbolo especial (?)
        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            # enumerate(f) nos da (número_de_línea, contenido_de_línea) para cada línea.
            for i, line in enumerate(f):
                # Solo queremos ver las primeras 15 líneas de cada archivo.
                if i >= 15:
                    break
                # repr() muestra la línea incluyendo caracteres especiales visibles,
                # como \n (salto de línea) o \t (tabulación). Limitamos a 200 caracteres
                # para no saturar la pantalla si las líneas son muy largas.
                print(repr(line[:200]))
    except Exception as e:
        # Si el archivo no se puede leer como texto, mostramos el error y continuamos.
        print(f'No se pudo leer como texto: {e}')

## 6. EDA básica por archivo

In [10]:
# Contamos el número de líneas de cada archivo para tener una idea del volumen de datos.
# Más líneas generalmente significa más mensajes (aunque depende del formato).
for _, row in df_inv.iterrows():
    fpath = RAW_DIR / row['archivo']
    try:
        # Abrimos el archivo y contamos sus líneas de forma eficiente.
        # En vez de cargar todo en memoria, usamos un generador:
        # "sum(1 for _ in f)" recorre línea a línea contando de 1 en 1 sin guardarlas.
        # El guion bajo "_" significa "no me importa el valor, solo cuento iteraciones".
        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            n_lines = sum(1 for _ in f)
        # Imprimimos el número de líneas alineado a la derecha (>8) para facilitar la lectura.
        print(f"{n_lines:>8} líneas  {row['archivo']}")
    except Exception as e:
        # Si el archivo no se puede leer, mostramos el error y continuamos con el siguiente.
        print(f'         ERROR   {row["archivo"]}: {e}')

## 7. Conclusiones

Completar manualmente tras ejecutar las celdas anteriores:

| Fuente | Archivo | Formato detectado | N líneas | Parser a usar |
|---|---|---|---|---|
| Jabber 2020 | ... | TXT / XML / JSON | ... | parse_txt / parse_jabber_xml / parse_rocketchat_json |
| Jabber 2021-2022 | ... | TXT / XML / JSON | ... | ... |
| Rocket.Chat | ... | TXT / XML / JSON | ... | ... |